<div style="font-size: 24px; line-height: 1.6;">

# Garbage In, Garbage Out & Poisoned Data

## A single stray row can poison a whole column

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Takeaway

Functions introduced: `pd.read_csv`, `head`, `tail`, `sample`, `shape`, `columns`, `info`, `dtypes`, `select_dtypes`, `pd.to_numeric`.

**Concept learned: inspect and clean the data before you analyze it.**

</div>

<div style="font-size: 24px; line-height: 1.6;">

### Imports

</div>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

<div style="font-size: 24px; line-height: 1.6;">

## The story

Before any analysis, open the file and look at it. A single leftover row — like a stray `test` row — can quietly change a numeric column into text. This notebook builds the habit of inspecting and cleaning *before* you compute anything.

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 1. Load the evidence with `pd.read_csv()`

</div>

In [2]:
df = pd.read_csv("../data/always_plot_demo.csv")

<div style="font-size: 24px; line-height: 1.6;">

## 2. First glance with `df.head()`

</div>

In [3]:
df.head()

,dataset,x,y
0,line,0.0,21.218868319017727
1,line,0.7092198581560284,16.230134497023833
2,line,1.4184397163120568,23.78194662719746
3,line,2.127659574468085,24.932471631522304
4,line,2.8368794326241136,13.756142933327915


<div style="font-size: 24px; line-height: 1.6;">

## 3. Last glance with `df.tail()`

Catch weird endings, appended notes, or format changes. **Look closely at the very last row here** — a stray `test, test, test` row sneaked in at the bottom of the file. This is exactly the kind of junk that `head()` would never show you.

</div>

In [4]:
df.tail()

,dataset,x,y
564,clusters,63.62597581254986,65.04914829033976
565,clusters,73.90984271491956,76.78298509193101
566,clusters,68.4492471898844,67.46717224040961
567,clusters,74.53239283496632,66.57174871969154
568,test,test,test


<div style="font-size: 24px; line-height: 1.6;">

That one bad row has a hidden cost. Because `x` and `y` now contain the text `"test"`, pandas could not read those columns as numbers — watch what `dtypes` says about them in a moment. A single garbage row at the edge poisons the type of the whole column.

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 4. Random glance with `df.sample()`

</div>

In [5]:
df.sample(5, random_state=42)

,dataset,x,y
204,curve,43.97163120567376,62.17025550062411
70,line,49.64539007092199,42.202219245655414
131,line,92.90780141843972,65.5925461899166
431,clusters,28.702136494585723,26.488763531271083
540,clusters,78.76207505995636,56.96708392730242


<div style="font-size: 24px; line-height: 1.6;">

## 5. How much evidence? `df.shape`

</div>

In [6]:
df.shape

(569, 3)

<div style="font-size: 24px; line-height: 1.6;">

## 6. Name the variables with `df.columns`

</div>

In [7]:
list(df.columns)

['dataset', 'x', 'y']

<div style="font-size: 24px; line-height: 1.6;">

## 7. Schema check with `df.info()`

</div>

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   dataset  569 non-null    str  
 1   x        569 non-null    str  
 2   y        569 non-null    str  
dtypes: str(3)
memory usage: 35.6 KB


<div style="font-size: 24px; line-height: 1.6;">

## 8. Data types with `df.dtypes`

Here is the proof. `x` and `y` should be `float64`, but they show up as `object` (text) — all because of that one `test` row.

</div>

In [9]:
df.dtypes

dataset    str
x          str
y          str
dtype: object

<div style="font-size: 24px; line-height: 1.6;">

## 9. Isolate numeric columns with `df.select_dtypes()`

Watch the consequence: asking for the numeric columns returns **nothing useful** — `x` and `y` are missing, because pandas no longer sees them as numbers.

</div>

In [10]:
numeric = df.select_dtypes(include="number")
numeric.head()

""
0
1
2
3
4


<div style="font-size: 24px; line-height: 1.6;">

## 10. Clean the bad row, then fix the types

Drop the junk row and convert `x` and `y` back to numbers. Now the columns are usable again.

</div>

In [11]:
df = df[df["dataset"] != "test"].copy()
df["x"] = pd.to_numeric(df["x"])
df["y"] = pd.to_numeric(df["y"])
df.dtypes

dataset        str
x          float64
y          float64
dtype: object

<div style="font-size: 24px; line-height: 1.6;">

## More ways data gets messy or poisoned

The `test` row was one kind of poison. Here are five more you will meet constantly in real datasets. Each uses a tiny made-up table so the problem is easy to see.

</div>

<div style="font-size: 24px; line-height: 1.6;">

### 1. Missing values in disguise

Real missing values are often hidden as sentinels like `-999`, `"N/A"`, or `"unknown"`. Pandas does not count them as `NaN` until you tell it to — so `isna()` reports the data as complete.

</div>

In [12]:
disguised = pd.DataFrame({
    "age": [34, -999, 28, 41],
    "city": ["Paris", "N/A", "Lima", "unknown"],
})
print("isna sees nothing wrong:", disguised.isna().sum().sum())
real = disguised.replace([-999, "N/A", "unknown"], pd.NA)
print("after replacing sentinels:", real.isna().sum().sum())

isna sees nothing wrong: 0
after replacing sentinels: 3


<div style="font-size: 24px; line-height: 1.6;">

### 2. Numbers stored as text

A column of prices like `"$1,200"` is text, not numbers. Summing it *glues the strings together* instead of adding.

</div>

In [13]:
prices = pd.DataFrame({"price": ["$1,200", "$950", "$3,400"]})
print("Broken (string) sum:", prices["price"].sum())
clean = prices["price"].str.replace(r"[$,]", "", regex=True).astype(float)
print("Real total:", clean.sum())

Broken (string) sum: $1,200$950$3,400
Real total: 5550.0


<div style="font-size: 24px; line-height: 1.6;">

### 3. Inconsistent categories

Casing and stray spaces split one real category into several. `"USA"`, `"usa"`, and `" USA "` look identical to us but are different groups to pandas.

</div>

In [14]:
survey = pd.DataFrame({"country": ["USA", "usa", " USA ", "Canada", "canada"]})
print("raw:    ", survey["country"].value_counts().to_dict())
normalized = survey["country"].str.strip().str.upper()
print("cleaned:", normalized.value_counts().to_dict())

raw:     {'USA': 1, 'usa': 1, ' USA ': 1, 'Canada': 1, 'canada': 1}
cleaned: {'USA': 3, 'CANADA': 2}


<div style="font-size: 24px; line-height: 1.6;">

### 4. Duplicate rows

A row copied twice silently double-counts. Always check `duplicated()` before trusting a total.

</div>

In [15]:
orders = pd.DataFrame({"order_id": [1, 2, 2, 3], "amount": [10, 25, 25, 8]})
print("with duplicates -> rows:", len(orders), "total:", orders["amount"].sum())
deduped = orders.drop_duplicates()
print("after dedupe    -> rows:", len(deduped), "total:", deduped["amount"].sum())

with duplicates -> rows: 4 total: 68
after dedupe    -> rows: 3 total: 43


<div style="font-size: 24px; line-height: 1.6;">

### 5. Impossible / out-of-range values

An `age` of `999` or `-3` is not a person — it is a typo or a sentinel, and one bad value can drag the mean far off. Range-check before you summarize.

</div>

In [16]:
people = pd.DataFrame({"age": [27, 5, 999, -3, 44]})
print("mean with bad rows:", round(people["age"].mean(), 1))
valid = people[(people["age"] >= 0) & (people["age"] <= 120)]
print("mean after range check:", round(valid["age"].mean(), 1))

mean with bad rows: 214.4
mean after range check: 25.3


<div style="font-size: 24px; line-height: 1.6;">

### A few more to watch for

- **Mixed or ambiguous date formats** (`01/02/03`) — parse with `pd.to_datetime` and check the result.
- **ID columns losing leading zeros** when read as numbers (`007` becomes `7`).
- **Text-encoding gremlins** (`café` showing up as `cafÃ©`).
- **Silently mixed units** (kg vs lb, USD vs EUR) in one column.
- **Trailing spaces in column names** (`"age "` vs `"age"`).

</div>